In [2]:
from manim import *
import numpy as np
from manim.mobject.geometry.boolean_ops import Difference, Intersection
import random
from Moleculas_Colidindo import *
import cv2
import tempfile
from MF_Tools import *

c:\Users\Enzo\AppData\Local\anaconda3\Lib\site-packages\pydub\utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)


In [15]:
%%manim -qh -v WARNING ViscosidadeCisalhamento


class ViscosidadeCisalhamento(Scene):
    def construct(self):
        # --- Configurações ---
        LARGURA_PLACA = 6
        ALTURA_PLACA = 0.4
        ALTURA_FLUIDO = 3.5
        COR_PLACA = GRAY_D
        COR_FLUIDO_CAMADA = BLUE_C 
        NUM_CAMADAS = 1 
        DESLOCAMENTO_TOTAL = 3 
        TEMPO_ANIMACAO = 5
        
        # --- 1. Criar as Placas ---
        base = Rectangle(width=LARGURA_PLACA, height=ALTURA_PLACA, color=COR_PLACA, fill_opacity=1)
        base.to_edge(DOWN, buff=1)
        
        topo = Rectangle(width=LARGURA_PLACA, height=ALTURA_PLACA, color=COR_PLACA, fill_opacity=1)
        topo.next_to(base, UP, buff=ALTURA_FLUIDO)
        
        # --- 2. Criar o Fluido (Estado Inicial - Retângulos) ---
        camadas_fluido = VGroup()
        altura_camada = ALTURA_FLUIDO / NUM_CAMADAS
        
        # Ponto de referência Y (topo da base)
        y_inicial = base.get_top()[1]
        x_centro = base.get_center()[0]
        x_esq = x_centro - LARGURA_PLACA / 2
        x_dir = x_centro + LARGURA_PLACA / 2

        # Lista para armazenar as camadas para referência posterior
        lista_camadas = []

        for i in range(NUM_CAMADAS):
            y_baixo = y_inicial + i * altura_camada
            y_cima = y_inicial + (i + 1) * altura_camada
            
            # Definindo os 4 vértices do retângulo manualmente
            # Ordem: [Inferior-Esq, Inferior-Dir, Superior-Dir, Superior-Esq]
            pontos = [
                [x_esq, y_baixo, 0],
                [x_dir, y_baixo, 0],
                [x_dir, y_cima, 0],
                [x_esq, y_cima, 0]
            ]
            
            camada = Polygon(*pontos, color=COR_FLUIDO_CAMADA, fill_opacity=0.6, stroke_width=1, stroke_color=WHITE)
            camadas_fluido.add(camada)
            lista_camadas.append(camada)

        # Textos e Labels
        label_base = Text("v = 0", font_size=24).next_to(base, DOWN)
        label_topo = MathTex("v = V_{max}", font_size=34).next_to(topo, UP)

        self.add(base, topo, camadas_fluido, label_base, label_topo)
        self.wait(1)

        # --- 3. Cálculo do Estado Final (Cisalhamento) ---
        
        animacoes = []
        
        # 3.1 Animar a Placa e Texto
        animacoes.append(topo.animate.shift(RIGHT * DESLOCAMENTO_TOTAL))
        animacoes.append(label_topo.animate.shift(RIGHT * DESLOCAMENTO_TOTAL))
        
        # 3.2 Animar a Deformação das Camadas
        for i, camada in enumerate(lista_camadas):
            # Calcula o deslocamento para a parte DE BAIXO desta camada
            fator_baixo = i / NUM_CAMADAS
            desloc_baixo = DESLOCAMENTO_TOTAL * fator_baixo
            
            # Calcula o deslocamento para a parte DE CIMA desta camada
            fator_cima = (i + 1) / NUM_CAMADAS
            desloc_cima = DESLOCAMENTO_TOTAL * fator_cima
            
            # Recupera as coordenadas originais
            # A ordem dos vértices no Polygon é mantida: IE, ID, SD, SE
            pontos_originais = camada.get_vertices()
            
            # Cria os novos pontos somando o deslocamento no eixo X
            novos_pontos = [
                pontos_originais[0] + [desloc_baixo, 0, 0], # Inf. Esq muda pouco
                pontos_originais[1] + [desloc_baixo, 0, 0], # Inf. Dir muda pouco
                pontos_originais[2] + [desloc_cima, 0, 0],  # Sup. Dir muda muito
                pontos_originais[3] + [desloc_cima, 0, 0]   # Sup. Esq muda muito
            ]
            
            # Transforma a camada atual para a nova forma (Paralelogramo)
            animacoes.append(
                camada.animate.become(
                    Polygon(*novos_pontos, color=COR_FLUIDO_CAMADA, fill_opacity=0.6, stroke_width=1, stroke_color=WHITE)
                )
            )

        # --- 4. Executar Animação ---
        self.play(*animacoes, run_time=TEMPO_ANIMACAO, rate_func=smooth)
        
        self.wait(2)

Manim Community v0.19.0

In [42]:
%%manim -qh -v WARNING ViscosidadeFluxoInfinitoMeio

import numpy as np
from manim import *

def create_h2o_molecule(molecule_radius=0.1):
    oxygen = Dot(radius=molecule_radius/2, color=RED).shift(UP*molecule_radius/2)
    h1 = Dot(radius=molecule_radius/2, color=WHITE).shift(LEFT*molecule_radius/2)
    h2 = Dot(radius=molecule_radius/2, color=WHITE).shift(RIGHT*molecule_radius/2)
    molecule = VGroup(oxygen, h1, h2)
    return molecule

class ViscosidadeFluxoInfinitoMeio(MovingCameraScene):
    def construct(self):
        # ==================================================================
        # PARTE 1: CONFIGURAÇÃO E ANIMAÇÃO DO GRÁFICO
        # ==================================================================
        
        # --- Configurações ---
        TUBE_LENGTH = 14
        TUBE_HEIGHT = 4.0
        TUBE_CENTER_Y = 0
        HALF_HEIGHT = TUBE_HEIGHT / 2
        
        ALTURA_PAREDE = 0.5
        COR_PAREDE = GRAY_D
        V_MAX = 2.5 
        NUM_PARTICLES = 600 
        MOLECULE_RADIUS = 0.03
        
        COR_GRAFICO = BLUE_A 
        COR_DESTAQUE = YELLOW 
        ESPESSURA_LINHA = 1.5      
        RAIO_PONTO = 0.03       
        VISUAL_SCALE = 0.5 
        
        # --- Tubo ---
        top_wall = Rectangle(width=TUBE_LENGTH, height=ALTURA_PAREDE, color=COR_PAREDE, fill_opacity=1)
        top_wall.move_to(UP * (HALF_HEIGHT + ALTURA_PAREDE/2))
        bottom_wall = Rectangle(width=TUBE_LENGTH, height=ALTURA_PAREDE, color=COR_PAREDE, fill_opacity=1)
        bottom_wall.move_to(DOWN * (HALF_HEIGHT + ALTURA_PAREDE/2))
        tube = VGroup(top_wall, bottom_wall)
        self.add(tube)

        # --- Física ---
        def get_u(y):
            y_rel = y - TUBE_CENTER_Y
            if abs(y_rel) > HALF_HEIGHT: return 0
            return V_MAX * (1 - (y_rel / HALF_HEIGHT)**2)

        def get_u_derivative(y):
            y_rel = y - TUBE_CENTER_Y
            return -2 * (V_MAX / (HALF_HEIGHT**2)) * y_rel

        # --- Partículas ---
        particles = VGroup()
        for _ in range(NUM_PARTICLES):
            x = np.random.uniform(-TUBE_LENGTH/2, TUBE_LENGTH/2)
            y = np.random.uniform(-HALF_HEIGHT + MOLECULE_RADIUS, HALF_HEIGHT - MOLECULE_RADIUS)
            mol = create_h2o_molecule(molecule_radius=MOLECULE_RADIUS)
            mol.move_to([x, y, 0])
            
            # --- ROTAÇÃO ALEATÓRIA ---
            # Define uma velocidade de rotação aleatória entre 2 e 6 radianos por segundo
            # Multiplica por 1 ou -1 para variar a direção (horário/anti-horário)
            mol.rotation_speed = np.random.uniform(2, 6) * np.random.choice([-1, 1])
            
            particles.add(mol)
        self.add(particles)

        def update_particles(mob, dt):
            for mol in mob:
                x, y, z = mol.get_center()
                velocity_x = get_u(y)
                
                # Movimento de Translação
                mol.shift(RIGHT * velocity_x * dt)
                
                # --- APLICAÇÃO DA ROTAÇÃO ---
                # Gira a molécula em torno do seu próprio centro
                mol.rotate(mol.rotation_speed * dt, about_point=mol.get_center())
                
                # Loop Infinito
                if mol.get_center()[0] > TUBE_LENGTH/2:
                    new_y = np.random.uniform(-HALF_HEIGHT + 0.05, HALF_HEIGHT - 0.05)
                    mol.move_to([-TUBE_LENGTH/2, new_y, 0])
                    
        particles.add_updater(update_particles)
        self.wait(18)

        # --- Zoom ---
        frame = self.camera.frame
        target_width = 6 
        target_center = [-3, -HALF_HEIGHT +1.36, 0] 
        self.play(frame.animate.move_to(target_center).set_width(target_width), run_time=2)
        self.wait(4)
        
        # --- Gráfico ---
        x_start_profile = -4.5 
        num_arrows = 6 
        vectors = VGroup()
        dots = VGroup() 
        eixo_y = Line(start=[x_start_profile, -HALF_HEIGHT, 0], end=[x_start_profile, 0, 0], color=COR_GRAFICO, stroke_width=ESPESSURA_LINHA)
        envelope_points = []
        for i in range(num_arrows + 1):
            y_pos = -HALF_HEIGHT + (i * (HALF_HEIGHT / num_arrows))
            vel_real = get_u(y_pos)
            arrow_length = vel_real * VISUAL_SCALE
            envelope_points.append([x_start_profile + arrow_length, y_pos, 0])
            if arrow_length > 0.05: 
                arrow = Arrow(start=[x_start_profile, y_pos, 0], end=[x_start_profile + arrow_length, y_pos, 0], buff=0, color=COR_GRAFICO, stroke_width=ESPESSURA_LINHA, tip_length=0.12, max_tip_length_to_length_ratio=0.2)
                vectors.add(arrow)
            dot = Dot(point=[x_start_profile, y_pos, 0], color=COR_GRAFICO, radius=RAIO_PONTO)
            dots.add(dot)
        envelope = VMobject().set_color(COR_GRAFICO).set_stroke(width=ESPESSURA_LINHA)
        envelope.set_points_smoothly(envelope_points)
        u_label = MathTex("u(y)", color=COR_GRAFICO, font_size=24).next_to(envelope, UP)
        no_slip_text = MathTex("u = 0", color=COR_GRAFICO, font_size=20)
        no_slip_text.next_to(dots[0], DOWN, buff=0.1) 
        self.play(Create(eixo_y), run_time=0.5)
        self.play(LaggedStart(*[GrowArrow(v) for v in vectors], lag_ratio=0.1), Create(dots), run_time=1)
        self.play(Create(envelope), Write(no_slip_text), Write(u_label))
        self.wait(1)

        # --- Tangente ---
        y_tracker = ValueTracker(-HALF_HEIGHT)
        moving_dot = Dot(color=COR_DESTAQUE, radius=0.04)
        tangent_line = Line(LEFT, RIGHT, color=COR_DESTAQUE, stroke_width=1.2).set_length(1.5)
        slope_label = MathTex(r"\frac{du}{dy}", color=COR_DESTAQUE, font_size=22)
        
        def update_tangent_elements(mob):
            current_y = y_tracker.get_value()
            u_val = get_u(current_y)
            visual_x = x_start_profile + (u_val * VISUAL_SCALE)
            point_location = np.array([visual_x, current_y, 0])
            moving_dot.move_to(point_location)
            tangent_line.move_to(point_location)
            slope_label.next_to(moving_dot, RIGHT, buff=0.2).shift(DOWN * 0.3)
            deriv = get_u_derivative(current_y)
            visual_dx = deriv * VISUAL_SCALE
            visual_dy = 1
            angle = np.arctan2(visual_dy, visual_dx)
            tangent_line.set_angle(angle)
        
        grupo_tangente = VGroup(moving_dot, tangent_line, slope_label)
        update_tangent_elements(grupo_tangente)
        
        self.wait(14)
        grupo_tangente.add_updater(lambda m: update_tangent_elements(m))
        self.add(grupo_tangente)
        self.play(y_tracker.animate.set_value(0), run_time=3, rate_func=linear)
        self.wait(0.5)
        self.play(y_tracker.animate.set_value(-HALF_HEIGHT), run_time=3, rate_func=smooth)
        grupo_tangente.clear_updaters() 
        self.wait(1)

        # ==================================================================
        # PARTE 2: FLUXO INFINITO (Seamless Loop)
        # ==================================================================

        # 1. Dar FadeOut no gráfico
        self.play(
            FadeOut(no_slip_text),
            FadeOut(grupo_tangente),
            FadeOut(u_label),
            FadeOut(envelope),
            FadeOut(dots),
            FadeOut(vectors),
            FadeOut(eixo_y),
            run_time=1.5
        )

        # Parar e remover partículas antigas
        particles.remove_updater(update_particles)

        # 2. Configuração das Camadas "Infinitas"
        COR_FLUIDO_CAMADA = BLUE_C
        COR_PADRAO = BLUE_A 
        NUM_CAMADAS = 10 
        ALTURA_FLUIDO = TUBE_HEIGHT
        altura_camada = ALTURA_FLUIDO / NUM_CAMADAS
        
        LARGURA_PADRAO = 10 
        ESPACO_CHEVRON = 1.0 
        SPEED_FACTOR = 0.2 

        camadas_group = VGroup()
        
        # Definição das camadas do meio
        indices_meio = [2, 3]

        for i in range(NUM_CAMADAS):
            y_center = -HALF_HEIGHT + (i + 0.5) * altura_camada
            
            # 2.1 Fundo da Camada
            camada_fundo = Rectangle(
                width=LARGURA_PADRAO, 
                height=altura_camada,
                color=COR_FLUIDO_CAMADA,
                fill_opacity=0.5,
                stroke_width=1,
                stroke_color=BLUE_B
            ).move_to([target_center[0], y_center, 0]) 
            
            # Lógica: Se for camada do meio, adiciona setas
            if i in indices_meio:
                chevrons = VGroup()
                num_chevrons = int(LARGURA_PADRAO / ESPACO_CHEVRON) + 2
                
                for k in range(num_chevrons):
                    x_pos = -LARGURA_PADRAO/2 + k * ESPACO_CHEVRON
                    p1 = [-0.05, 0.1, 0]
                    p2 = [0.05, 0, 0]
                    p3 = [-0.05, -0.1, 0]
                    
                    seta = VMobject().set_points_as_corners([p1, p2, p3])
                    seta.set_stroke(color=COR_PADRAO, width=6, opacity=0.2)
                    seta.scale_to_fit_height(altura_camada * 0.6)
                    seta.shift([x_pos, 0, 0])
                    chevrons.add(seta)
                
                chevrons.move_to(camada_fundo.get_center())
                camadas_group.add(camada_fundo, chevrons)
                
                # Updater de Movimento
                velocidade = get_u(y_center) * SPEED_FACTOR
                
                def gerar_updater(mob, vel, espaco):
                    def updater(m, dt):
                        m.shift(RIGHT * vel * dt)
                        distancia_percorrida = m.get_center()[0] - target_center[0]
                        if distancia_percorrida > espaco/2: 
                             m.shift(LEFT * espaco)
                    return updater

                upd = gerar_updater(chevrons, velocidade, ESPACO_CHEVRON)
                chevrons.add_updater(upd)
            
            else:
                camadas_group.add(camada_fundo)

        # 3. Transição e Execução
        self.play(
            FadeOut(particles),
            FadeIn(camadas_group),
            run_time=1.5
        )
        
        # ==================================================================
        # PARTE 3: TENSÃO DE CISALHAMENTO (Múltiplos Pares)
        # ==================================================================
        
        # Interface entre camada 2 e 3
        idx_interface = 2 
        y_interface = -HALF_HEIGHT + (idx_interface + 1) * altura_camada
        
        shear_color = RED
        shear_len = 0.7
        tip_len = 0.12   
        tip_height = 0.08
        offset_y = 0.03  
        stroke_w = 1.8   

        # Distribuição com 3 pontos
        x_positions = np.linspace(-4.5, -1.5, 3) 
        
        all_shear_arrows = VGroup()

        for cx in x_positions:
            # 3.1 Seta Superior (Para a Direita)
            p_start_top = [cx - shear_len/2, y_interface + offset_y, 0]
            p_end_top =   [cx + shear_len/2, y_interface + offset_y, 0]
            line_top = Line(p_start_top, p_end_top, color=shear_color, stroke_width=stroke_w)
            
            tip_top_pts = [
                [p_end_top[0], p_end_top[1], 0], 
                [p_end_top[0] - tip_len, p_end_top[1] + tip_height, 0], 
                [p_end_top[0] - tip_len, p_end_top[1], 0] 
            ]
            tip_top = Polygon(*tip_top_pts, color=shear_color, fill_opacity=1, stroke_width=0)
            shear_top = VGroup(line_top, tip_top)

            # 3.2 Seta Inferior (Para a Esquerda)
            p_start_bot = [cx + shear_len/2, y_interface - offset_y, 0]
            p_end_bot =   [cx - shear_len/2, y_interface - offset_y, 0]
            line_bot = Line(p_start_bot, p_end_bot, color=shear_color, stroke_width=stroke_w)
            
            tip_bot_pts = [
                [p_end_bot[0], p_end_bot[1], 0], 
                [p_end_bot[0] + tip_len, p_end_bot[1] - tip_height, 0], 
                [p_end_bot[0] + tip_len, p_end_bot[1], 0] 
            ]
            tip_bot = Polygon(*tip_bot_pts, color=shear_color, fill_opacity=1, stroke_width=0)
            shear_bottom = VGroup(line_bot, tip_bot)
            
            all_shear_arrows.add(VGroup(shear_top, shear_bottom))

        shear_label = MathTex(r"\tau", color=RED, font_size=40).next_to(all_shear_arrows, UP, buff=0)

        self.wait(3)
        self.play(
            LaggedStart(*[GrowFromCenter(s) for s in all_shear_arrows], lag_ratio=0.1),
            Write(shear_label),
            run_time=2
        )
        
        self.wait(11)

Manim Community v0.19.0

In [52]:
%%manim -qh -v WARNING Viscosidade

from manim import *

class Viscosidade(Scene):
    def construct(self):
        # Monta a equação com cada parte separada
        cisalhamento = MathTex(
            r"\tau", "=", r"\mu", r"\frac{du}{dy}",
            font_size=72
        )

        self.play(Write(cisalhamento), run_time=1.2)
        self.wait(2.5)

        # Seleciona o du/dy (índice 3)
        dudY = cisalhamento[3]

        # Wiggle no du/dy
        self.play(Wiggle(dudY, scale_value=1.3))
        self.play(dudY.animate.set_color(YELLOW))
        self.wait(2)

        # O μ é o terceiro elemento (índice 2)
        mu = cisalhamento[2]

        # Wiggle no μ
        self.play(Wiggle(mu, scale_value=1.3))

        # Cor azul apenas no μ
        self.play(mu.animate.set_color(BLUE_C))
        self.wait(0.5)

        # Texto explicativo azul
        texto_viscosidade = Tex(
            "viscosidade",
            font_size=68,
            color=BLUE_C
        )
        texto_viscosidade.next_to(mu, DOWN, buff=1.5)

        # Seta azul mais fina
        arrow = Arrow(
            start=mu.get_bottom(),
            end=texto_viscosidade.get_top(),
            buff=0.1,
            color=BLUE_C,
            stroke_width=2
        )

        self.play(
            GrowArrow(arrow),
            Write(texto_viscosidade)
        )

        self.wait(6)


Manim Community v0.19.0

In [36]:
%%manim -qh -v WARNING fimViscosidade
class fimViscosidade(Scene):
    def construct(self):
        TituloViscosidade = Tex("Viscosidade", font_size=56)
        self.play(Write(TituloViscosidade))
        self.wait(2)
        self.play(FadeOut(TituloViscosidade))

Manim Community v0.19.0